# 05 — Predictive Modeling: Calibrated Churn, Discounted Value, and Expected Profit

**Owner:** M5 — Thư  
**Purpose:** run the professional, script-backed M5 pipeline.

This notebook is intentionally thin. The reusable production logic lives in `scripts/modeling.py`; this notebook is only for execution and review.

## What this notebook does

1. Validates the current repo structure and core input files.
2. Runs the M5 v3 end-to-end pipeline from `config/paths.yaml`.
3. Produces calibrated churn probabilities, two-part discounted 60-day value estimates, SHAP explainability outputs, seasonality audit files, and expected incremental profit rankings.
4. Keeps M4 unchanged; M5 consumes the delivered feature table as-is.

In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Data').exists() and (PROJECT_ROOT.parent / 'Data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Config:', PROJECT_ROOT / 'config' / 'paths.yaml')

Project root: c:\Users\Thuww\DDM-Churn-Project
Config: c:\Users\Thuww\DDM-Churn-Project\config\paths.yaml


## 1. Validate core inputs

In [6]:
from scripts.data_processing import validate_core_inputs

input_checks = validate_core_inputs(PROJECT_ROOT / 'config' / 'paths.yaml')
input_checks

{
  "feature_table_csv": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\models\\final_ML_features.csv",
    "exists": true,
    "size_bytes": 690607
  },
  "transaction_master_parquet": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\transactions_master.parquet",
    "exists": true,
    "size_bytes": 24332905
  },
  "customer_base_parquet": {
    "path": "C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\customer_base_labeled.parquet",
    "exists": true,
    "size_bytes": 120755
  }
}


{'feature_table_csv': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models\\final_ML_features.csv',
  'exists': True,
  'size_bytes': 690607},
 'transaction_master_parquet': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\transactions_master.parquet',
  'exists': True,
  'size_bytes': 24332905},
 'customer_base_parquet': {'path': 'C:\\Users\\Thuww\\DDM-Churn-Project\\Data\\Processed\\customer_base_labeled.parquet',
  'exists': True,
  'size_bytes': 120755}}

## 2. Run M5 pipeline

In [7]:
from scripts.modeling import run_m5_pipeline

summary = run_m5_pipeline(PROJECT_ROOT / 'config' / 'paths.yaml')
summary

[M5] Project root: C:\Users\Thuww\DDM-Churn-Project
[M5] Tuning classifier: Logistic Regression balanced
[M5]   Logistic Regression balanced: CV candidate 1/10
[M5]   Logistic Regression balanced: CV candidate 2/10
[M5]   Logistic Regression balanced: CV candidate 3/10
[M5]   Logistic Regression balanced: CV candidate 4/10
[M5]   Logistic Regression balanced: CV candidate 5/10
[M5]   Logistic Regression balanced: CV candidate 6/10
[M5]   Logistic Regression balanced: CV candidate 7/10
[M5]   Logistic Regression balanced: CV candidate 8/10
[M5]   Logistic Regression balanced: CV candidate 9/10
[M5]   Logistic Regression balanced: CV candidate 10/10
[M5] Tuning classifier: Random Forest balanced
[M5] Tuning classifier: Extra Trees balanced
[M5] Tuning classifier: XGBoost weighted
[M5]   XGBoost weighted: CV candidate 1/8


C:\Users\Thuww\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:318: UserWarning: The total space of parameters 8 is smaller than n_iter=10. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[M5]   XGBoost weighted: CV candidate 2/8
[M5]   XGBoost weighted: CV candidate 3/8
[M5]   XGBoost weighted: CV candidate 4/8
[M5]   XGBoost weighted: CV candidate 5/8
[M5]   XGBoost weighted: CV candidate 6/8
[M5]   XGBoost weighted: CV candidate 7/8
[M5]   XGBoost weighted: CV candidate 8/8
[M5] Fitting future-active model: Active Logistic Regression
[M5] Fitting future-active model: Active Random Forest
[M5] Fitting conditional discounted value model: Ridge Regression
[M5] Fitting conditional discounted value model: Random Forest Regressor
[M5] Fitting conditional discounted value model: XGBoost Regressor
[M5] Pipeline complete.
{
  "version": "v3_discounted_two_part_value_shap_seasonality",
  "project_root": "C:\\Users\\Thuww\\DDM-Churn-Project",
  "feature_rows": 2499,
  "churn_rate": 0.1208483393357343,
  "champion_churn_model": "Logistic Regression balanced",
  "calibration_method": "isotonic",
  "champion_threshold": 0.06999999999999999,
  "test_PR_AUC_calibrated": 0.3120009403

{'version': 'v3_discounted_two_part_value_shap_seasonality',
 'project_root': 'C:\\Users\\Thuww\\DDM-Churn-Project',
 'feature_rows': 2499,
 'churn_rate': 0.1208483393357343,
 'champion_churn_model': 'Logistic Regression balanced',
 'calibration_method': 'isotonic',
 'champion_threshold': 0.06999999999999999,
 'test_PR_AUC_calibrated': 0.3120009403015265,
 'test_F2_score_calibrated': 0.4690831556503198,
 'test_brier_score_calibrated': 0.09320834579658996,
 'active_model': 'Active Random Forest__isotonic',
 'value_model': 'Random Forest Regressor',
 'annual_discount_rate': 0.08,
 'base_profitable_customers': 0,
 'shap_status_file': 'C:\\Users\\Thuww\\DDM-Churn-Project\\reports\\internal_briefs\\M5_shap_status.json',
 'outputs_dir': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models'}

## 3. Review generated outputs

In [8]:
import pandas as pd

model_dir = PROJECT_ROOT / 'models'
metrics = pd.read_csv(model_dir / 'model_metrics.csv')
active_metrics = pd.read_csv(model_dir / 'active_model_metrics.csv')
value_metrics = pd.read_csv(model_dir / 'value_model_metrics.csv')
scenario_summary = pd.read_csv(model_dir / 'scenario_profit_summary.csv')
calibration_summary = pd.read_csv(model_dir / 'calibration_summary.csv')

display(metrics.head())
display(calibration_summary.head())
display(active_metrics.head())
display(value_metrics.head())
display(scenario_summary)


,model,tuned,features_used,best_cv_PR_AUC,best_params,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,...,test_threshold,test_predicted_positive_rate,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,test_TN,test_FP,test_FN,test_TP,best_cv_PR_AUC_std
0,Logistic Regression balanced,True,numeric+categorical,0.315523,"{""model__C"": 0.005}",0.366964,0.747265,0.483559,0.183150,0.819672,...,0.43,0.484,0.453930,0.12,0.333930,242,198,16,44,0.044709
1,XGBoost weighted,True,numeric+categorical,0.366348,"{""model__colsample_bytree"": 0.85, ""model__lear...",0.340226,0.748516,0.496084,0.273381,0.622951,...,0.47,0.252,0.430301,0.12,0.310301,349,91,25,35,0.066313
2,Random Forest balanced,False,numeric+categorical,NaN,{},0.297267,0.741850,0.491453,0.205357,0.754098,...,0.09,0.490,0.111580,0.12,-0.008420,239,201,16,44,NaN
3,Extra Trees balanced,False,numeric+categorical,NaN,{},0.279860,0.712517,0.474041,0.211055,0.688525,...,0.09,0.436,0.113680,0.12,-0.006320,261,179,21,39,NaN
4,Dummy prior,False,none,NaN,{},0.122000,0.500000,0.409946,0.122000,1.000000,...,0.01,1.000,0.120747,0.12,0.000747,0,440,0,60,NaN


,champion_model,calibration_method,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,val_brier_score,val_threshold,val_predicted_positive_rate,...,test_brier_score,test_threshold,test_predicted_positive_rate,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,test_TN,test_FP,test_FN,test_TP
0,Logistic Regression balanced,isotonic,0.369740,0.771799,0.498008,0.193798,0.819672,0.088233,0.07,0.516,...,0.093208,0.07,0.458,0.120913,0.12,0.000913,255,185,16,44
1,Logistic Regression balanced,sigmoid,0.366964,0.747265,0.476636,0.175258,0.836066,0.096066,0.10,0.582,...,0.094899,0.10,0.534,0.121605,0.12,0.001605,219,221,14,46
2,Logistic Regression balanced,raw_uncalibrated,0.366964,0.747265,0.483559,0.183150,0.819672,0.205896,0.43,0.546,...,0.205190,0.43,0.484,0.453930,0.12,0.333930,242,198,16,44


,active_model,calibration_method,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,val_brier_score,val_threshold,val_predicted_positive_rate,...,test_brier_score,test_threshold,test_predicted_positive_rate,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,test_TN,test_FP,test_FN,test_TP
0,Active Random Forest,isotonic,0.985062,0.897769,0.980562,0.90982,1.0,0.059024,0.01,0.998,...,0.069763,0.01,1.0,0.901128,0.892,0.009128,0,54,0,446
1,Active Logistic Regression,isotonic,0.983525,0.889102,0.980562,0.90982,1.0,0.061413,0.01,0.998,...,0.075268,0.01,1.0,0.903431,0.892,0.011431,0,54,0,446
2,Active Random Forest,raw_uncalibrated,0.984976,0.881105,0.980562,0.90982,1.0,0.063950,0.40,0.998,...,0.070056,0.40,1.0,0.887688,0.892,-0.004312,0,54,0,446
3,Active Random Forest,sigmoid,0.984976,0.881105,0.980562,0.90982,1.0,0.068567,0.53,0.998,...,0.077319,0.53,1.0,0.904518,0.892,0.012518,0,54,0,446
4,Active Logistic Regression,raw_uncalibrated,0.983875,0.868991,0.980562,0.90982,1.0,0.068618,0.12,0.998,...,0.072582,0.12,1.0,0.891017,0.892,-0.000983,0,54,0,446


,value_model,target,val_RMSE_log,val_MAE_log,val_R2_log,val_MAE_revenue,test_RMSE_log,test_MAE_log,test_R2_log,test_MAE_revenue
0,Random Forest Regressor,log1p(discounted_future_revenue_60d | active),0.894759,0.666764,0.501525,141.410733,0.893071,0.644547,0.488270,144.231453
1,Ridge Regression,log1p(discounted_future_revenue_60d | active),0.918765,0.686662,0.474418,166.320471,0.957195,0.713307,0.412145,191.129702
2,XGBoost Regressor,log1p(discounted_future_revenue_60d | active),0.923122,0.690962,0.469421,162.355861,0.920081,0.681295,0.456848,168.642975


,scenario,scenario_type,gross_margin,save_rate_given_treatment,treatment_cost,profitable_customer_count,profitable_customer_share,total_expected_incremental_profit_if_target_positive_only,top_30pct_risk_customer_count,total_expected_incremental_profit_if_target_top_30pct_risk
0,conservative,named,0.20,0.03,5.0,0,0.0000,0.000000,750,-3627.520595
1,base,named,0.25,0.05,5.0,0,0.0000,0.000000,750,-3494.834573
2,optimistic,named,0.30,0.08,5.0,2,0.0008,5.564033,750,-3260.082380


## 4. Handoff files for M6

Use `models/high_risk_customers_for_ab_test.csv` as the experiment population candidate file.

Use `models/profitable_treatment_candidates_base.csv` only as the profitability sanity-check file under base assumptions.

Important interpretation:
- `p_churn_calibrated` is the churn-risk score used for business formulas.
- `predicted_discounted_value_60d_if_active` is a discounted 60-day value estimate, not full lifetime CLV.
- `predicted_expected_discounted_value_60d` comes from the two-part model: `p_future_active × value_if_active`.
- SHAP files explain prediction drivers, not causal treatment effects.